### We'll start with a reliable benchmark for EXP_001 (Recursive) and EXP_002_v2 (Structure-based). First, we'll compare dense similarity and MMR across the same 973 test questions. Then we'll add:
1. Scheme-level metrics: HitRate/Recall@K, Precision@K, MRR@K and NDCG@K.
2. Retrieval performance: latency, duplicate chunks, unique schemes and diversity.
3. Ragas: supplementary evaluation of retrieved context and answer quality where the ground truth supports it.
4. Langfuse: traces and experiment-level observability.

In [ ]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

DATA_DIR = PROJECT_ROOT / "notebooks" / "golden_dataset" / "final" / "Final_dataset_splits"

def load_json_dataset(filename):
    file_path = DATA_DIR / filename

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return pd.DataFrame(data)

train_df = load_json_dataset("train.json")
validation_df = load_json_dataset("validation.json")
test_df = load_json_dataset("test.json")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

display(test_df.head(3))

Train: (4539, 12)
Validation: (973, 12)
Test: (973, 12)


,test_id,faq_id,scheme_id,scheme_name,question,ground_truth_answer,categories,level,source_url,relevant_scheme_ids,relevant_doc_ids,answerable
0,GD_000031,0Be1F5gB5uF80_MGnI9U_FAQ_001,0Be1F5gB5uF80_MGnI9U,Dayanand Bandodkar Scheme For Higher Education...,Is this scheme limited to certain fields of st...,"The scheme covers a wide range of fields, incl...",[Education & Learning],State,https://www.myscheme.gov.in/schemes/dbsheo,[0Be1F5gB5uF80_MGnI9U],[],True
1,GD_004164,dxfCF5gB5uF80_MGAJJb_FAQ_006,dxfCF5gB5uF80_MGAJJb,PMMSY: Construction of Fish Retail Markets inc...,Is there a limit to how many units of central ...,"Yes, central assistance is restricted to one u...","[Agriculture,Rural & Environment, Business & E...",State,https://www.myscheme.gov.in/schemes/pmmsycrmhr,[dxfCF5gB5uF80_MGAJJb],[],True
2,GD_001489,FBfZF5gB5uF80_MGUpOf_FAQ_020,FBfZF5gB5uF80_MGUpOf,Accidental Benefit,What form needs to be used to apply for this w...,The application must be made in the prescribed...,[Social welfare & Empowerment],State,https://www.myscheme.gov.in/schemes/adwb,[FBfZF5gB5uF80_MGUpOf],[],True


In [3]:
assert len(train_df) == 4539
assert len(validation_df) == 973
assert len(test_df) == 973

assert test_df["test_id"].is_unique
assert test_df["question"].notna().all()
assert test_df["relevant_scheme_ids"].notna().all()

print("Dataset validation passed.")
print("Unique test schemes:", test_df["scheme_id"].nunique())
print("Answerable:", test_df["answerable"].sum())
print("Unanswerable:", (~test_df["answerable"]).sum())

Dataset validation passed.
Unique test schemes: 473
Answerable: 973
Unanswerable: 0


In [4]:
from pathlib import Path
from langchain_chroma import Chroma
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

BASE_VS_PATH = Path("G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison")

EXP001_PATH = BASE_VS_PATH / "exp_001"
EXP002_PATH = BASE_VS_PATH / "exp_002_v2"

# Same embedding model and configuration used during indexing
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

# Check paths before connecting
assert EXP001_PATH.exists(), f"Missing: {EXP001_PATH}"
assert EXP002_PATH.exists(), f"Missing: {EXP002_PATH}"

# # Connect to existing Chroma indexes
# client = chromadb.PersistentClient(
#     path="G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison\\exp_001"
# )

# collection_001 = client.get_collection("exp_001_recursive")

# Connect to existing collections
vs_exp001 = Chroma(
    persist_directory=str(EXP001_PATH),
    collection_name="exp_001_recursive",
    embedding_function=embedding_function
)

vs_exp002 = Chroma(
    persist_directory=str(EXP002_PATH),
    collection_name="exp_002_structure_v2",
    embedding_function=embedding_function
)

print("EXP_001 documents:", vs_exp001._collection.count())
print("EXP_002 documents:", vs_exp002._collection.count())

g:\SEVA_Ai\SEVA_Ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 556.76it/s]


EXP_001 documents: 124029
EXP_002 documents: 91858


In [5]:
def retrieve_documents(
    vectorstore,
    query,
    method="similarity",
    k=10,
    fetch_k=50,
    lambda_mult=0.5
):
    if method == "similarity":

        results = vectorstore.similarity_search_with_score(
            query,
            k=k
        )

        documents = []

        for rank, (doc, distance) in enumerate(results, start=1):
            documents.append({
                "rank": rank,
                "scheme_id": doc.metadata.get("scheme_id"),
                "section": doc.metadata.get("section"),
                "document_type": doc.metadata.get("document_type"),
                "text": doc.page_content,
                "distance": float(distance)
            })

    elif method == "mmr":

        results = vectorstore.max_marginal_relevance_search(
            query,
            k=k,
            fetch_k=fetch_k,
            lambda_mult=lambda_mult
        )

        documents = []

        for rank, doc in enumerate(results, start=1):
            documents.append({
                "rank": rank,
                "scheme_id": doc.metadata.get("scheme_id"),
                "section": doc.metadata.get("section"),
                "document_type": doc.metadata.get("document_type"),
                "text": doc.page_content,
                "distance": None
            })

    else:
        raise ValueError(
            "method must be 'similarity' or 'mmr'"
        )

    return documents

In [6]:
query = test_df.iloc[0]["question"]

experiments = {
    "EXP001_similarity": (vs_exp001, "similarity"),
    "EXP001_mmr": (vs_exp001, "mmr"),
    "EXP002_similarity": (vs_exp002, "similarity"),
    "EXP002_mmr": (vs_exp002, "mmr")
}

smoke_results = {}

for experiment_name, (vectorstore, method) in experiments.items():

    results = retrieve_documents(
        vectorstore=vectorstore,
        query=query,
        method=method,
        k=10,
        fetch_k=50,
        lambda_mult=0.5
    )

    smoke_results[experiment_name] = results

    print(f"\n{'=' * 70}")
    print(experiment_name)
    print(f"{'=' * 70}")

    display(pd.DataFrame(results)[[
        "rank",
        "scheme_id",
        "section",
        "document_type",
        "distance"
    ]])


EXP001_similarity


,rank,scheme_id,section,document_type,distance
0,1,0Be1F5gB5uF80_MGnI9U,faq,faq,0.237890
1,2,7sdChZwB0AGc0F0iwAOR,faq,faq,0.531097
2,3,3KG-F5gBXu-iUKXqaXaN,faq,faq,0.723972
3,4,sxe5F5gB5uF80_MGwZDl,faq,faq,0.731396
4,5,fcbBF5gB0AGc0F0i_zYi,faq,faq,0.740569
5,6,iceVwpwB0AGc0F0i0AXi,faq,faq,0.752161
6,7,phe0F5gB5uF80_MGFY_C,faq,faq,0.786338
7,8,FaJ7vJwBXu-iUKXqf0nf,faq,faq,0.803402
8,9,F8ZMU5kB0AGc0F0i3Tnv,faq,faq,0.804978
9,10,ahiA3JwB5uF80_MG-mEs,faq,faq,0.813412



EXP001_mmr


,rank,scheme_id,section,document_type,distance
0,1,0Be1F5gB5uF80_MGnI9U,faq,faq,None
1,2,7sdChZwB0AGc0F0iwAOR,faq,faq,None
2,3,FaJ7vJwBXu-iUKXqf0nf,faq,faq,None
3,4,F8ZMU5kB0AGc0F0i3Tnv,faq,faq,None
4,5,yhgn0pwB5uF80_MGH2BB,faq,faq,None
5,6,zKKZGZwBXu-iUKXqbEam,details,scheme_section,None
6,7,pxf9F5gB5uF80_MGZJNo,faq,faq,None
7,8,LKKm9ZwBXu-iUKXqX0xI,details,scheme_section,None
8,9,M6E9GJgBXu-iUKXqBXnT,details,scheme_section,None
9,10,y8b9F5gB0AGc0F0iYzfT,faq,faq,None



EXP002_similarity


,rank,scheme_id,section,document_type,distance
0,1,0Be1F5gB5uF80_MGnI9U,faq,faq,0.237890
1,2,7sdChZwB0AGc0F0iwAOR,faq,faq,0.531097
2,3,3KG-F5gBXu-iUKXqaXaN,faq,faq,0.723972
3,4,sxe5F5gB5uF80_MGwZDl,faq,faq,0.731396
4,5,fcbBF5gB0AGc0F0i_zYi,faq,faq,0.740569
5,6,zKKZGZwBXu-iUKXqbEam,brief_description,scheme_section,0.751699
6,7,iceVwpwB0AGc0F0i0AXi,faq,faq,0.752161
7,8,phe0F5gB5uF80_MGFY_C,faq,faq,0.786338
8,9,FaJ7vJwBXu-iUKXqf0nf,faq,faq,0.803402
9,10,F8ZMU5kB0AGc0F0i3Tnv,faq,faq,0.804978



EXP002_mmr


,rank,scheme_id,section,document_type,distance
0,1,0Be1F5gB5uF80_MGnI9U,faq,faq,None
1,2,7sdChZwB0AGc0F0iwAOR,faq,faq,None
2,3,zKKZGZwBXu-iUKXqbEam,brief_description,scheme_section,None
3,4,FaJ7vJwBXu-iUKXqf0nf,faq,faq,None
4,5,F8ZMU5kB0AGc0F0i3Tnv,faq,faq,None
5,6,yhgn0pwB5uF80_MGH2BB,faq,faq,None
6,7,pRe9F5gB5uF80_MG4JH5,faq,faq,None
7,8,pxf9F5gB5uF80_MGZJNo,faq,faq,None
8,9,DhdZe5kB5uF80_MGkZV-,brief_description,scheme_section,None
9,10,y8b9F5gB0AGc0F0iYzfT,faq,faq,None
